# 👤 **OpenCV & HOG**
<hr> Adapted for <a href="https://mmla.gse.harvard.edu/">ez-mmla</a>, <a href="http://lit.gse.harvard.edu/">Harvard LITLab</a> (Schneider et al.)
<br><br>
<!-- Ideally, put a GIF or an image here that best portrays the demo or the model. -->
<center> <img width="600px" src="https://drive.google.com/uc?id=10Kz3hBCcUwIXCv0bzZAK0Gy-u6xtPZ77">
</center>
<center>The results obtained from the model </center>

> This is an opensourse project from Data Flair that ulitlize OpenCV an HOG to detect and count human.

**Some notes:**
* Sometimes, the model is not very accurate.


## 🔧 **Setup**
<hr>
⏳ <b>Colab/Colab Pro</b>: 5 minutes


In [ ]:
#@title #### **Install and import dependencies**

from IPython.display import display, HTML, clear_output

completed_actions = []
current_action = ""

def start_step(action):
  global current_action
  current_action = action
  display(HTML(f"<b style='font-size: 15px'>{action} ⏳</b>"))
  display(HTML("<hr>"))

def finished():
  global current_action, completed_actions
  current_action = ""
  display(HTML(f"<b style='font-size: 15px; color: green;'>Done ✅ </b>"))
  completed_actions = []

def end_step():
  global current_action, completed_actions
  completed_actions.append(current_action)
  current_action = ""
  clear_output()
  for action in completed_actions:
    display(HTML(f"<b style='font-size: 15px'>{action} ✅ </b>"))
    display(HTML("<hr>"))

start_step("Install opencv")
!pip install opencv-python --q
end_step()

start_step("Install imutils")
!pip install imutils --q
end_step()

start_step("Install pytube")
!pip install pytube --q
end_step()

start_step("Install imageio")
!pip install imageio==2.4.1 --q
end_step()

start_step("Import necessary libraries")
from IPython.display import display, Javascript,HTML, Image, Video
from google.colab.output import eval_js
from base64 import b64decode,b64encode
from google.colab import files
import numpy as np
from scipy.io.wavfile import read as wav_read
import io
from scipy.io.wavfile import write
import os
from google.colab import drive
from pytube import YouTube
from moviepy.editor import *
import cv2
import imutils
import numpy as np
import argparse

end_step()

start_step("Load the  model")

end_step()

finished()

## 🕹 **Demo**
<hr>

> This model allows for both image and video processing, so choose your input below!

In [4]:
#@title #### **Image/video input**
#@markdown Click run and choose an option for the image or the video input.

import ipywidgets as widgets
from google.colab import files
from IPython.display import Video, Image
from pytube import YouTube
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode,b64encode
import cv2

input_file = None
image_or_video = "image"

# The function takes in a dictionary of options and their corresponding functions
def dropdown_menu(options, on_finished):
    clear_output()
    output = widgets.Output()
    dropdown = widgets.Dropdown(options = options.keys(), description='Actions:')

    def on_option_change(choice):
        global input_file
        clear_output()
        # Run the associated function
        input_file = options[choice]()
        clear_output()
        display(input_widgets)
        on_finished(input_file)

    dropdown.observe(lambda e: on_option_change(e.new), names='value')
    input_widgets = widgets.HBox([dropdown])
    display(input_widgets)

    # Initially run the first option (the demo)
    on_option_change(list(options.keys())[0])

# This function runs whenever the action user chose finishes.
def on_finished(fn):
  global image_or_video
  if fn:
    display(HTML(f"<hr><b>Below is a preview of the file:</b> <code>{fn}</code><br><br>"))
    if fn[-4:] in ['.mp4', ".mov", ".avi"] or image_or_video == "video":
      image_or_video = "video"
      display(Video(fn, embed=True, width=400))
    else:
      image_or_video = "image"
      display(Image(fn, embed=True, width=400))

# Functions corresponding to the actions the user selected!
def demo():
  global image_or_video
  image_or_video = "image"

  youtube = YouTube('https://www.youtube.com/shorts/IZHyckC8FRw')
  stream=youtube.streams.get_highest_resolution()
  videofile=stream.download()
  vidcap = cv2.VideoCapture(videofile)
  success,image = vidcap.read()
  file="/content/image.jpg"
  cv2.imwrite(file, image)
  return file

def take_photo(filename='photo.jpg', quality=0.8):
  global image_or_video
  image_or_video = "image"

  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

def record_video():
  global image_or_video
  image_or_video = "video"

  js=Javascript("""
    async function recordVideo() {
      const options = { mimeType: "video/webm; codecs=vp9" };
      const div = document.createElement('div');
      const capture = document.createElement('button');
      const stopCapture = document.createElement("button");

      capture.textContent = "Start Recording";
      capture.style.background = "orange";
      capture.style.color = "white";

      stopCapture.textContent = "Stop Recording";
      stopCapture.style.background = "red";
      stopCapture.style.color = "white";
      div.appendChild(capture);

      const video = document.createElement('video');
      const recordingVid = document.createElement("video");
      video.style.display = 'block';

      const stream = await navigator.mediaDevices.getUserMedia({audio:true, video: true});

      let recorder = new MediaRecorder(stream, options);
      document.body.appendChild(div);
      div.appendChild(video);

      video.srcObject = stream;
      video.muted = true;

      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => {
        capture.onclick = resolve;
      });
      recorder.start();
      capture.replaceWith(stopCapture);

      await new Promise((resolve) => stopCapture.onclick = resolve);
      recorder.stop();
      let recData = await new Promise((resolve) => recorder.ondataavailable = resolve);
      let arrBuff = await recData.data.arrayBuffer();

      // stop the stream and remove the video element
      stream.getVideoTracks()[0].stop();
      div.remove();

      let binaryString = "";
      let bytes = new Uint8Array(arrBuff);
      bytes.forEach((byte) => {
        binaryString += String.fromCharCode(byte);
      })
    return btoa(binaryString);
    }
  """)
  display(js)
  data=eval_js('recordVideo({})')
  binary=b64decode(data)
  with open("/content/recorded_video.mp4","wb") as video_file:
    video_file.write(binary)
  return "/content/recorded_video.mp4"

def drive_file_chooser(exts):
  # This function renders a file chooser which finds all the files of valid extensions
  # inside your Google Drive and allows the user to select the file
  files = []
  FILE_HTML = ""

  JS = """
  <style>
  .list-item {
    background: #eee;
    border-radius: 5px;
    padding: 5px;
    transition: all ease 0.1s;
    width: 50%;
    margin: 5px;
  }
  .list-item:hover {
    background: #ddd;
    transition: all ease 0.1s;
  }
  </style>
  <p> Below are the files obtained from your Google Drive. </p>
  <b> Choose a file from the list below. </b>
  <hr>
  <div> [!!!REPLACE!!!] </div>

  <script>
    var changestate;
    var selection = new Promise(resolve => {
      changestate = function(i) {
        var el = document.getElementById(i)
        el.style.backgroundColor = '#b0ceff';
        el.style.fontWeight = "900";
        resolve(i);
      }
    });
  </script>
  """
  i = 0
  for root, directories, filenames in os.walk("/content/drive/My Drive"):
    for filename in filenames:
      # change the extensions to the type of file we are looking for
      if filename.endswith(tuple(exts)):
          FILE_HTML += """<div id="{}" class="list-item" onclick="changestate('{}')">{}</div>""".format(i, i, filename)
          i += 1
          files.append(os.path.join(root,filename))
  display(HTML(JS.replace("[!!!REPLACE!!!]", FILE_HTML)))
  selection = eval_js("selection")
  return files[int(selection)]

def google_drive():
  from google.colab import drive
  display(HTML("<i>Please wait while we fetch files from your Google Drive...</i>"))
  drive.mount('/content/drive')
  clear_output()
  return drive_file_chooser(["png", "jpg", "mp4", "avi", "mov"])

def upload():
  try:
    uploaded = files.upload()
    fn = list(uploaded.keys())[0]
    return "/content/" + fn
  except:
    return None

dropdown_menu({
    "Demo picture": demo,
    "Take a picture": take_photo,
    "Record video": record_video,
    "Upload file": upload,
    "Upload from Google Drive": google_drive,
}, on_finished)


AgeRestrictedError: IZHyckC8FRw is age restricted, and can't be accessed without logging in.

In [ ]:
#@title #### **Run the demo**
#@markdown Run this cell to run the model.

from google.colab import files
start_step("Downloading model")
!gdown https://drive.google.com/u/2/uc?id=1rYn0dTNAGoZZlxE6ty33zF6T2ydoQMxx&export=download&confirm=t&uuid=eb618fa9-1292-4a13-95c6-b4c18d5c64a3&at=ANzk5s5gpfMwMiPunqH1ehZRbrUQ:1680466289719
end_step()
if image_or_video == 'image':
  !python human-counting-project-code.py -i $input_file -o result.jpg -csv result.csv
  display(Image('/content/result.jpg'))
else:
  !python human-counting-project-code.py -v $input_file -o /content/result.mp4 -csv /content/result.csv
  display(Video('/content/result.mp4', embed=True, width=400))



In [ ]:
#@title #### **Save results**
#@markdown If you would like to save your results (a video/image and a csv file), press run.
if image_or_video == 'image':
    files.download('/content/result.jpg')
else:
    files.download('/content/result.mp4')
files.download('/content/result.csv')

## 📄 **Documentation**
<hr>

* https://data-flair.training/blogs/python-project-real-time-human-detection-counting/
